# Hormuz Cable Risk
## Coastal Infrastructure Vulnerability Index for Submarine Cable Landing Points in the Gulf Zone

This notebook documents the full analysis pipeline: data sources, methodology, results, and limitations.

**Data coverage:** 1989 – March 2026  
**Author:** Carolina Cruz  
**Repository:** [github.com/CarolinaMCruz/hormuz-cable-risk](https://github.com/CarolinaMCruz/hormuz-cable-risk)

## Abstract

This project develops a **Coastal Infrastructure Vulnerability Index** for submarine cable landing points in the Gulf zone, covering ten countries: Yemen, Pakistan, Iraq, Kuwait, Saudi Arabia, Iran, United Arab Emirates, Qatar, Bahrain, and Oman.

Using data from TeleGeography's Submarine Cable Map, the UCDP Georeferenced Event Dataset (1989–2026), and Global Fishing Watch AIS vessel presence data, the index quantifies each country's exposure to digital isolation through four components: conflict exposure near landing points (35%), dependency on Gulf-zone cables (30%), inverse cable redundancy (20%), and landing point concentration (15%).

Results show Yemen and Pakistan as the most structurally vulnerable countries. A network analysis reveals that FALCON is Iran's only cable with global reach outside the Gulf corridor — if damaged, Iran loses connectivity to 184 of 185 countries. A comparison of AIS vessel presence data between pre-conflict (Dec 2025–Jan 2026) and active conflict (Feb–Mar 2026) periods shows a 34% reduction in maritime traffic, with FALCON remaining the most exposed cable in both periods.

---

## Resumen

Este proyecto desarrolla un Índice de Vulnerabilidad de Infraestructura Costera para los puntos de aterrizaje de cables submarinos en la zona del Golfo, cubriendo diez países: Yemen, Pakistán, Iraq, Kuwait, Arabia Saudita, Irán, Emiratos Árabes Unidos, Qatar, Bahréin y Omán.

Utilizando datos del Mapa de Cables Submarinos de TeleGeography, el UCDP Georeferenced Event Dataset (1989–2026) y datos de presencia AIS de Global Fishing Watch, el índice cuantifica la exposición de cada país al aislamiento digital mediante cuatro componentes: exposición a conflicto cerca de los puntos de aterrizaje (35%), dependencia de cables en la zona del Golfo (30%), redundancia inversa de cables (20%) y concentración de puntos de aterrizaje (15%).

El análisis de red revela que FALCON es el único cable de Irán con alcance global fuera del corredor del Golfo. Si se daña, Irán pierde conectividad con 184 de 185 países. Una comparación de datos AIS entre el período pre-conflicto y el conflicto activo muestra una reducción del 34% en el tráfico marítimo.

## 1. Context

The Strait of Hormuz is the world's most critical maritime chokepoint for oil transport. Less discussed is its role as a submarine cable corridor connecting the Gulf states to Europe and Asia.

Submarine cable landing points — where cables transition from sea to land — represent the most vulnerable segment of this infrastructure. They are physically accessible, geographically concentrated, and surrounded by support facilities that can be disrupted by nearby conflict.

This project asks: **which countries in the Gulf zone have the most exposed cable landing infrastructure, and how does nearby conflict increase their vulnerability?**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load processed datasets
df_cables    = pd.read_csv('../data/processed/cables.csv')
df_landing   = pd.read_csv('../data/processed/landing_points.csv')
df_conflicto = pd.read_csv('../data/processed/ucdp_gulf_zone.csv', low_memory=False)
df_indice    = pd.read_csv('../data/processed/indice_riesgo.csv')
df_ais_p1    = pd.read_csv('../data/processed/ais_periodo1.csv')
df_ais_p2    = pd.read_csv('../data/processed/ais_periodo2.csv')
df_exp_p1    = pd.read_csv('../data/processed/cables_exposicion_periodo1.csv')
df_exp_p2    = pd.read_csv('../data/processed/cables_exposicion_periodo2.csv')

print(f"Cables: {len(df_cables)}")
print(f"Landing points: {len(df_landing)}")
print(f"Conflict events (Gulf zone): {len(df_conflicto):,}")
print(f"Countries in index: {len(df_indice)}")
print(f"AIS records Period 1 (pre-conflict): {len(df_ais_p1):,}")
print(f"AIS records Period 2 (active conflict): {len(df_ais_p2):,}")

## 2. Infrastructure Overview

### 2.1 Submarine Cable Landing Points in the Gulf Zone

In [ ]:
ZONA = [
    "Iran", "Oman", "Yemen", "Saudi Arabia",
    "United Arab Emirates", "Qatar", "Kuwait", "Bahrain",
    "Pakistan", "Iraq"
]

lp_zona = df_landing[df_landing["country"].isin(ZONA)]
conteo = lp_zona["country"].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
conteo.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Landing Points per Country in the Gulf Zone", fontsize=14, pad=15)
ax.set_xlabel("Country")
ax.set_ylabel("Number of Landing Points")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

print(f"\nTotal landing points in zone: {len(lp_zona)}")
print(conteo.to_string())

### 2.2 Cables per Country

In [ ]:
cables_pais = (
    lp_zona.groupby("country")["cable_id"]
    .nunique()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 5))
cables_pais.plot(kind="bar", ax=ax, color="darkorange", edgecolor="white")
ax.set_title("Unique Submarine Cables per Country in the Gulf Zone", fontsize=14, pad=15)
ax.set_xlabel("Country")
ax.set_ylabel("Number of Cables")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

print(f"\nCables per country:")
print(cables_pais.to_string())

### 2.3 Conflict Events in the Gulf Zone (1989–2026)

In [ ]:
df_conflicto["date_start"] = pd.to_datetime(df_conflicto["date_start"])
df_conflicto["year"] = df_conflicto["date_start"].dt.year
df_conflicto["country"] = df_conflicto["country"].replace({"Yemen (North Yemen)": "Yemen"})

eventos_año = df_conflicto.groupby("year").size()

fig, ax = plt.subplots(figsize=(14, 5))
eventos_año.plot(kind="line", ax=ax, color="crimson", linewidth=2)
ax.fill_between(eventos_año.index, eventos_año.values, alpha=0.2, color="crimson")
ax.set_title("Conflict Events in the Gulf Zone per Year (1989–2026)", fontsize=14, pad=15)
ax.set_xlabel("Year")
ax.set_ylabel("Number of Events")
ax.axvline(x=2003, color="gray", linestyle="--", alpha=0.7, label="Iraq War (2003)")
ax.axvline(x=2014, color="orange", linestyle="--", alpha=0.7, label="ISIS rise (2014)")
ax.axvline(x=2015, color="blue", linestyle="--", alpha=0.7, label="Yemen War (2015)")
ax.set_ylim(bottom=0)
ax.legend()
plt.tight_layout()
plt.show()

### 2.4 Conflict Events by Country

In [ ]:
eventos_pais = df_conflicto["country"].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
eventos_pais.plot(kind="bar", ax=ax, color="crimson", edgecolor="white")
ax.set_title("Conflict Events per Country in the Gulf Zone (1989–2026)", fontsize=14, pad=15)
ax.set_xlabel("Country")
ax.set_ylabel("Number of Events")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

print(f"\nEvents per country:")
print(eventos_pais.to_string())

## 3. Methodology

### 3.1 Coastal Vulnerability Index

The **Digital Vulnerability Index** measures risk at cable landing points — not along submarine cable routes on the seafloor. Landing points represent the most vulnerable segment of any submarine cable system: they are physically accessible, geographically concentrated, and surrounded by support infrastructure.

The index is calculated per country using four weighted components:

| Component | Weight | Description |
|-----------|--------|-------------|
| Exposure | 35% | UCDP conflict events within 50km of cable landing points (2020–2026) |
| Dependency | 30% | Share of a country's cables that pass through the Gulf risk zone |
| Redundancy | 20% | Total number of cables serving the country (inverse) |
| Concentration | 15% | Maximum number of cables sharing a single landing point |

Each component is normalized to [0, 1] before weighting.

### 3.2 AIS Traffic Exposure Methodology

Global Fishing Watch AIS data is delivered as a grid of cells with `spatial_resolution=LOW` (0.1 degrees, approximately 11km x 11km). Each cell has a central point represented by latitude and longitude coordinates.

To determine whether a cable passes through a given cell, we take the central point of each cell and measure the minimum distance to the nearest cable geometry. If that distance is less than 6km (half the cell size), we assign the traffic from that cell to the cable.

This approach captures the most likely zone of interaction between vessels and cables, acknowledging that exact vessel positions within a cell are unknown. The result is an **exposure score per cable**: accumulated vessel-hours of maritime traffic in cells where each cable is present.

## 4. Digital Vulnerability Index

### 4.1 Risk Index by Country

In [ ]:
df_indice_sorted = df_indice.sort_values("indice_riesgo", ascending=True)

colors = []
for val in df_indice_sorted["indice_riesgo"]:
    if val >= 0.6:
        colors.append("#ff2222")
    elif val >= 0.45:
        colors.append("#ff8800")
    elif val >= 0.30:
        colors.append("#ffdd00")
    else:
        colors.append("#44bb44")

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(df_indice_sorted["country"], df_indice_sorted["indice_riesgo"],
               color=colors, edgecolor="white")
ax.set_title("Digital Vulnerability Index by Country", fontsize=14, pad=15)
ax.set_xlabel("Risk Index (0 = Low, 1 = High)")
ax.axvline(x=0.60, color="#ff2222", linestyle="--", alpha=0.5, label="High risk threshold")
ax.axvline(x=0.45, color="#ff8800", linestyle="--", alpha=0.5, label="Medium-high threshold")
ax.axvline(x=0.30, color="#ffdd00", linestyle="--", alpha=0.5, label="Medium threshold")
for bar, val in zip(bars, df_indice_sorted["indice_riesgo"]):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=10)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

### 4.2 Index Components Breakdown

In [ ]:
df_plot = df_indice.sort_values("indice_riesgo", ascending=False).copy()

df_plot["w_exposicion"]    = df_plot["n_exposicion"] * 0.35
df_plot["w_dependencia"]   = df_plot["n_dependencia"] * 0.30
df_plot["w_redundancia"]   = df_plot["n_redundancia"] * 0.20
df_plot["w_concentracion"] = df_plot["n_concentracion"] * 0.15

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(df_plot["country"], df_plot["w_exposicion"],    label="Exposure (35%)",      color="#cc2222")
ax.bar(df_plot["country"], df_plot["w_dependencia"],   label="Dependency (30%)",    color="#ff8800",
       bottom=df_plot["w_exposicion"])
ax.bar(df_plot["country"], df_plot["w_redundancia"],   label="Redundancy (20%)",    color="#4488ff",
       bottom=df_plot["w_exposicion"] + df_plot["w_dependencia"])
ax.bar(df_plot["country"], df_plot["w_concentracion"], label="Concentration (15%)", color="#44bb44",
       bottom=df_plot["w_exposicion"] + df_plot["w_dependencia"] + df_plot["w_redundancia"])
ax.set_title("Risk Index Components by Country", fontsize=14, pad=15)
ax.set_xlabel("Country")
ax.set_ylabel("Weighted Score")
ax.tick_params(axis="x", rotation=45)
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 5. Network Connectivity Analysis: What Happens if FALCON is Cut?

In [ ]:
import networkx as nx

# Build graph: countries as nodes, cables as edges
G = nx.Graph()
for cable_id, grupo in df_landing.groupby("cable_id"):
    paises = grupo["country"].unique().tolist()
    cable_name = grupo["cable_name"].iloc[0]
    for i in range(len(paises)):
        for j in range(i+1, len(paises)):
            if G.has_edge(paises[i], paises[j]):
                G[paises[i]][paises[j]]["cables"].append(cable_name)
                G[paises[i]][paises[j]]["weight"] += 1
            else:
                G.add_edge(paises[i], paises[j], cables=[cable_name], weight=1)

# Simulate FALCON cut
G_sin_falcon = G.copy()
edges_falcon = [(u, v) for u, v, d in G_sin_falcon.edges(data=True)
                if "FALCON" in d.get("cables", [])]
G_sin_falcon.remove_edges_from(edges_falcon)

componente = nx.node_connected_component(G_sin_falcon, "Iran")
print(f"Global network: {G.number_of_nodes()} countries")
print(f"With FALCON active - Iran directly connected to: {len(list(G.neighbors('Iran')))} countries")
print(f"Without FALCON - Iran isolated to: {len(componente)} country")
print(f"Countries out of Iran's reach: {G.number_of_nodes() - len(componente)}")

### Finding

FALCON is Iran's only cable with connections outside the Gulf corridor. If damaged, Iran loses connectivity to **184 of 185 countries** in the global submarine cable network. Its remaining 5 cables only connect to neighboring countries within the same risk zone — no alternative route exists outside the Strait of Hormuz.

This directly answers the project's original question: Iran faces **total digital isolation risk from a single cable failure**.

*Note: This models topological connectivity only, not actual traffic routing or BGP configurations.*

## 6. AIS Maritime Traffic Analysis

### 6.1 Overview: Pre-Conflict vs Active Conflict

We compare two periods using Global Fishing Watch AIS vessel presence data:
- **Period 1 (Pre-conflict):** December 2025 – January 2026
- **Period 2 (Active conflict):** February 2026 – March 2026

In [ ]:
# Overall comparison
horas_p1 = df_ais_p1["hours"].sum()
horas_p2 = df_ais_p2["hours"].sum()
cambio = (horas_p2 - horas_p1) / horas_p1 * 100

print("MARITIME TRAFFIC COMPARISON\n")
print(f"Period 1 (Pre-conflict):    {horas_p1:>12,.0f} vessel-hours")
print(f"Period 2 (Active conflict): {horas_p2:>12,.0f} vessel-hours")
print(f"Change:                     {cambio:>12.1f}%")

fig, ax = plt.subplots(figsize=(8, 5))
periodos = ["Pre-conflict\n(Dec 2025 - Jan 2026)", "Active conflict\n(Feb 2026 - Mar 2026)"]
horas = [horas_p1, horas_p2]
colores = ["#4488ff", "#ff4444"]
bars = ax.bar(periodos, horas, color=colores, edgecolor="white", width=0.5)
for bar, val in zip(bars, horas):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50000,
            f"{val:,.0f}", ha="center", fontsize=10)
ax.set_title("Total Vessel-Hours in Gulf Zone by Period", fontsize=14, pad=15)
ax.set_ylabel("Total Vessel-Hours")
ax.set_ylim(0, max(horas) * 1.15)
plt.tight_layout()
plt.show()

### 6.2 Cable Exposure to Maritime Traffic

In [ ]:
# Top 10 cables by traffic exposure - both periods
top_p1 = df_exp_p1.head(10).copy()
top_p2 = df_exp_p2.head(10).copy()

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, df, titulo, color in [
    (axes[0], top_p1, "Pre-conflict (Dec 2025 - Jan 2026)", "#4488ff"),
    (axes[1], top_p2, "Active conflict (Feb 2026 - Mar 2026)", "#ff4444")
]:
    nombres = [n[:35] + "..." if len(n) > 35 else n for n in df["cable_name"]]
    ax.barh(nombres[::-1], df["horas_trafico"][::-1], color=color, edgecolor="white")
    ax.set_title(titulo, fontsize=11, pad=10)
    ax.set_xlabel("Vessel-Hours")
    ax.tick_params(axis="y", labelsize=8)

fig.suptitle("Top 10 Cables by Maritime Traffic Exposure", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 6.3 Change in Cable Exposure Between Periods

In [ ]:
# Merge and calculate change
merged = df_exp_p1[["cable_id", "cable_name", "horas_trafico"]].merge(
    df_exp_p2[["cable_id", "horas_trafico"]], on="cable_id", suffixes=("_p1", "_p2")
)
merged["cambio_pct"] = ((merged["horas_trafico_p2"] - merged["horas_trafico_p1"]) /
                        merged["horas_trafico_p1"] * 100).round(1)
merged = merged.sort_values("cambio_pct")

top_reduccion = merged.head(10).copy()
nombres = [n[:40] + "..." if len(n) > 40 else n for n in top_reduccion["cable_name"]]

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(nombres[::-1], top_reduccion["cambio_pct"][::-1], color="#ff4444", edgecolor="white")
ax.set_title("Cables with Largest Traffic Reduction During Active Conflict", fontsize=13, pad=15)
ax.set_xlabel("Change in Vessel-Hours (%)")
ax.tick_params(axis="y", labelsize=9)
ax.axvline(x=0, color="gray", linewidth=0.8)
plt.tight_layout()
plt.show()

print("\nTop 10 cables with largest traffic reduction:")
print(merged[["cable_name", "horas_trafico_p1", "horas_trafico_p2", "cambio_pct"]].head(10).to_string(index=False))

### 6.4 Interpretation

- **34% overall reduction** in vessel-hours between pre-conflict and active conflict periods, suggesting vessels are avoiding or reducing time in the Gulf zone during the conflict.
- **FALCON remains the most exposed cable** in both periods (1.75M hours pre-conflict, 1.15M during conflict), confirming its strategic importance and vulnerability.
- **Tata TGN-Gulf (-50.6%) and Kuwait-Iran (-49.4%)** show the largest traffic reductions, suggesting avoidance of the inner Gulf routes specifically.
- Despite the reduction in traffic, the ranking of most-exposed cables remains largely stable, indicating that the same cables face the highest accident risk regardless of conflict intensity.

## 7. Sensitivity Analysis

How does the risk ranking change if we modify the component weights? We test three alternative weighting schemes against the baseline.

In [ ]:
baseline = {
    "n_exposicion":    0.35,
    "n_dependencia":   0.30,
    "n_redundancia":   0.20,
    "n_concentracion": 0.15,
}

escenarios = {
    "Baseline\n(Exp:35 Dep:30 Red:20 Con:15)": baseline,
    "Exposure Heavy\n(Exp:50 Dep:25 Red:15 Con:10)": {
        "n_exposicion": 0.50, "n_dependencia": 0.25,
        "n_redundancia": 0.15, "n_concentracion": 0.10,
    },
    "Dependency Heavy\n(Exp:20 Dep:50 Red:20 Con:10)": {
        "n_exposicion": 0.20, "n_dependencia": 0.50,
        "n_redundancia": 0.20, "n_concentracion": 0.10,
    },
    "Equal Weights\n(Exp:25 Dep:25 Red:25 Con:25)": {
        "n_exposicion": 0.25, "n_dependencia": 0.25,
        "n_redundancia": 0.25, "n_concentracion": 0.25,
    },
}

resultados = pd.DataFrame({"country": df_indice["country"]})
for nombre, pesos in escenarios.items():
    resultados[nombre] = sum(
        df_indice[col] * peso for col, peso in pesos.items()
    ).round(3)

fig, axes = plt.subplots(1, 4, figsize=(18, 6), sharey=False)
for ax, col in zip(axes, resultados.columns[1:]):
    sorted_df = resultados.sort_values(col, ascending=True)
    colors = ["#ff2222" if v >= 0.6 else "#ff8800" if v >= 0.45
              else "#ffdd00" if v >= 0.30 else "#44bb44"
              for v in sorted_df[col]]
    ax.barh(sorted_df["country"], sorted_df[col], color=colors, edgecolor="white")
    ax.set_title(col, fontsize=9)
    ax.set_xlim(0, 1)
    ax.tick_params(axis="y", labelsize=8)
fig.suptitle("Sensitivity Analysis: Risk Index Under Different Weight Schemes", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 7.1 Ranking Stability

In [ ]:
print("RANKING STABILITY ANALYSIS\n")
rankings = {}
for col in resultados.columns[1:]:
    rankings[col] = resultados.sort_values(col, ascending=False)["country"].tolist()

for pais in resultados["country"]:
    posiciones = [ranking.index(pais) + 1 for ranking in rankings.values()]
    min_pos = min(posiciones)
    max_pos = max(posiciones)
    variacion = max_pos - min_pos
    estabilidad = "STABLE" if variacion <= 1 else f"varies {min_pos}-{max_pos}"
    print(f"  {pais:<25} {estabilidad}")

### 7.2 Interpretation

**Robustly vulnerable (stable high ranking):** Yemen and Pakistan maintain top positions across all weighting schemes. Their vulnerability is structural, not a methodological artifact.

**Robustly resilient (stable low ranking):** Oman and Qatar remain consistently low-risk regardless of weights.

**Methodologically sensitive (unstable ranking):** Kuwait, Saudi Arabia, UAE, Iran, and Bahrain shift significantly depending on whether conflict exposure or cable dependency is prioritized. These countries have mixed risk profiles that require deeper case-by-case analysis.

This sensitivity analysis confirms that the index is robust for extreme cases but should be interpreted with caution for mid-range countries.

## 8. Temporal Analysis: How Has Risk Evolved? (2015–2026)

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

RADIO_M = 50 * 1000

df_coords = pd.read_csv('../data/processed/landing_points_coords.csv')
lp = df_landing[df_landing["country"].isin(ZONA)].merge(df_coords, on="landing_id", how="left")
lp = lp.dropna(subset=["latitude", "longitude"])
lp["geometry"] = lp.apply(lambda r: Point(r["longitude"], r["latitude"]), axis=1)
gdf_lp = gpd.GeoDataFrame(lp, crs="EPSG:4326").to_crs("EPSG:3857")

df_conflicto["geometry"] = df_conflicto.apply(
    lambda r: Point(r["longitude"], r["latitude"]), axis=1
)
gdf_conf = gpd.GeoDataFrame(df_conflicto, crs="EPSG:4326").to_crs("EPSG:3857")
gdf_conf["year"] = gdf_conf["date_start"].dt.year

print("Calculating exposure by year... (1-2 minutes)")
años = range(2015, 2027)
resultados_temporales = []

for año in años:
    conf_año = gdf_conf[gdf_conf["year"] == año]
    for pais in ZONA:
        lp_pais = gdf_lp[gdf_lp["country"] == pais]
        total_eventos = 0
        for _, lp_row in lp_pais.iterrows():
            buffer = lp_row["geometry"].buffer(RADIO_M)
            eventos = conf_año[conf_año["geometry"].within(buffer)]
            total_eventos += len(eventos)
        resultados_temporales.append({"year": año, "country": pais, "eventos": total_eventos})

df_temporal = pd.DataFrame(resultados_temporales)
print("Done.")

In [ ]:
paises_plot = ["Yemen", "Iraq", "Pakistan", "Iran", "Saudi Arabia", "Oman"]

fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=True)
axes = axes.flatten()

for ax, pais in zip(axes, paises_plot):
    data = df_temporal[df_temporal["country"] == pais].sort_values("year")
    ax.fill_between(data["year"], data["eventos"], alpha=0.3, color="crimson")
    ax.plot(data["year"], data["eventos"], color="crimson", linewidth=2)
    ax.set_title(pais, fontsize=12)
    ax.set_xlabel("Year")
    ax.set_ylabel("Conflict events")
    ax.set_xticks(range(2015, 2027, 2))
    ax.set_ylim(bottom=0)
    ax.tick_params(axis="x", rotation=45)

fig.suptitle("Conflict Exposure Near Cable Landing Points by Year (2015–2026)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 8.1 Temporal Findings

In [ ]:
print("TEMPORAL ANALYSIS - KEY OBSERVATIONS\n")
for pais in paises_plot:
    data = df_temporal[df_temporal["country"] == pais].sort_values("year")
    max_año = data.loc[data["eventos"].idxmax(), "year"]
    max_val = data["eventos"].max()
    total = data["eventos"].sum()
    print(f"{pais:<25} Peak: {max_año} ({max_val} events)  Total: {total}")

### 8.2 Interpretation

- **Yemen:** Peak in 2018 (329 events), coinciding with the most intense phase of the civil war and Houthi operations near the Red Sea coast. Cable landing points in Al Hudaydah were directly in the conflict zone.
- **Pakistan:** Peak in 2015 (414 events), driven by military operations in Baluchistan near the Karachi coastal area where submarine cables land.
- **Iran:** Only 13 total events across the entire period. This reflects a critical limitation: the Iran-Israel-US conflict (2025–2026) is primarily aerial and naval, not captured as terrestrial armed conflict in UCDP. Iran's actual risk is significantly underestimated by this dataset.
- **Iraq and Oman:** Zero events near landing points across all years. Iraq's conflict was inland; Oman has no significant armed conflict.

## 9. Key Findings

In [ ]:
print("=" * 55)
print("  KEY FINDINGS")
print("=" * 55)

top = df_indice.iloc[0]
print(f"\n1. Most vulnerable country: {top['country']}")
print(f"   Risk index: {top['indice_riesgo']}")
print(f"   Cables: {int(top['cables_totales'])} (all in risk zone)")
print(f"   Conflict events near cables: {int(top['eventos_totales'])}")

bottom = df_indice.iloc[-1]
print(f"\n2. Most resilient country: {bottom['country']}")
print(f"   Risk index: {bottom['indice_riesgo']}")
print(f"   Cables: {int(bottom['cables_totales'])} (high redundancy)")
print(f"   Conflict events near cables: {int(bottom['eventos_totales'])}")

print(f"\n3. Critical cables (pass through Iran or Yemen):")
criticos = df_landing[
    df_landing["country"].isin(["Iran", "Yemen"])
]["cable_name"].value_counts()
for cable, _ in criticos.head(5).items():
    print(f"   - {cable}")

print(f"\n4. Single points of failure:")
cables_por_lp = df_landing.groupby(["landing_id","landing_name","country"]).size().reset_index(name="n_cables")
spof = cables_por_lp[
    cables_por_lp["country"].isin(["Iran","Yemen","Iraq"]) &
    (cables_por_lp["n_cables"] >= 3)
].sort_values("n_cables", ascending=False)
print(spof[["landing_name","country","n_cables"]].to_string(index=False))

print(f"\n5. Maritime traffic reduction during conflict:")
horas_p1 = df_ais_p1["hours"].sum()
horas_p2 = df_ais_p2["hours"].sum()
cambio = (horas_p2 - horas_p1) / horas_p1 * 100
print(f"   Pre-conflict:    {horas_p1:,.0f} vessel-hours")
print(f"   Active conflict: {horas_p2:,.0f} vessel-hours")
print(f"   Change:          {cambio:.1f}%")

## 10. Conclusions

### What this analysis demonstrates

**1. Yemen and Pakistan are the most vulnerable countries in the Gulf zone** regardless of how the index components are weighted. This is a structural finding: both countries have few cables, all concentrated in the risk zone, with significant conflict activity near their coastal landing infrastructure.

**2. Oman is the most resilient country** despite being geographically closest to the Strait. High cable redundancy (20 cables) and absence of nearby conflict make it structurally resistant to digital isolation.

**3. FALCON is the single most critical cable in the region.** It is both the most exposed to maritime traffic (highest accident risk) and the only cable connecting Iran to the global internet outside the Gulf. If FALCON is damaged, Iran loses connectivity to 184 of 185 countries.

**4. Maritime traffic dropped 34% during the active conflict period** (Feb–Mar 2026 vs Dec 2025–Jan 2026), suggesting vessels are actively avoiding or reducing time in the Gulf zone. Despite this, FALCON remains the most exposed cable in both periods.

**5. The mid-range group (Kuwait, Saudi Arabia, UAE, Iran, Bahrain) shows methodological sensitivity.** Their rankings shift depending on component weights and require case-by-case analysis.

**6. Iran is likely underestimated by the conflict index.** UCDP captures verified terrestrial armed conflict. Iran’s actual risk includes naval operations, drone activity, proxy conflicts, and geopolitical pressure not reflected in the dataset.

### What this analysis cannot demonstrate

- Risk along submarine cable routes on the seafloor (requires higher-resolution AIS data)
- Risk from naval operations, cyberattacks, or sanctions-related disruptions
- Political or strategic motivations of any state actor
- Causal relationships between conflict and cable damage

### Final note on methodology

This index is a proxy measure of coastal infrastructure vulnerability, not a comprehensive geopolitical risk assessment. It should be interpreted alongside qualitative analysis of the regional context, particularly for countries like Iran where quantitative conflict data systematically underrepresents actual risk.

## 11. Limitations

- **Scope:** This index measures risk at cable landing points (coastal infrastructure), not along submarine routes on the seafloor. Seabed risk from vessel anchoring or fishing requires higher-resolution AIS vessel traffic data.
- **UCDP coverage:** Verified armed conflict events only. Naval operations, cyber attacks, and geopolitical pressure are not captured, likely underestimating Iran’s actual risk.
- **Cable geometry:** TeleGeography routes are cartographic representations, not precise engineering data.
- **Exposure radius:** 50km buffer around landing points for conflict index. 6km buffer for AIS-cable crossing.
- **AIS spatial resolution:** LOW resolution (0.1°, ~11km cells) limits precision of cable-traffic crossing analysis.
- **AISstream:** Real-time vessel tracking was tested but the service was unavailable at time of development. The capture script is included for future use.
- **GFW vessel presence:** FLAG grouping used instead of GEARTYPE, as GEARTYPE is not available for the presence dataset. This limits ability to distinguish vessel types (tankers, fishing boats, cargo).

## 12. Data Sources

| Source | Dataset | Coverage | Access |
|--------|---------|----------|--------|
| TeleGeography | Submarine Cable Map API v3 | Current | Public API |
| UCDP | GED Global v25.1 | 1989–2024 | Free download |
| UCDP | Candidate v25.01.25.12 | 2025 | Free download |
| UCDP | Candidate v26.0.1 | Jan 2026 | Free download |
| UCDP | Candidate v26.0.2 | Feb 2026 | Free download |
| Global Fishing Watch | AIS Vessel Presence (public-global-presence:v4.0) | Dec 2025–Mar 2026 | Free API (token required) |

## 13. References

### Academic
- Raleigh, C., Linke, A., Hegre, H., & Karlsen, J. (2010). Introducing ACLED: Armed Conflict Location and Event Data. *Journal of Peace Research, 47*(5), 651–660.
- Davies, S., Pettersson, T., & Öberg, M. (2023). Organized violence 1989–2022 and the return of conflicts between states. *Journal of Peace Research, 60*(4).

### Reports and Articles
- Crowley, J. (March 19, 2026). Hormuz submarine cables face heightened risk from vessel build-up. *AGBI*.
- International Cable Protection Committee. (2023). *Submarine cable faults: Causes and prevention*. ICPC Publication No. 2.
- TeleGeography. (2025). *The State of the Network 2026 Edition*.
- United Kingdom Maritime Trade Operations (UKMTO). (April 2, 2026). *Recent Incidents Summary (Feb 28 – Apr 2, 2026)*.

### Data
- TeleGeography. (2026). *Submarine Cable Map API v3*. https://www.submarinecablemap.com
- Uppsala Conflict Data Program. (2025–2026). *UCDP GED and Candidate Datasets*. https://ucdp.uu.se/downloads
- Global Fishing Watch. (2026). *AIS Vessel Presence API (public-global-presence:v4.0)*. https://globalfishingwatch.org/our-apis/